In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project location
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

# Load the Day 5 feature dataset
feature_path = (
    project_dir
    / "data"
    / "processed"
    / "Day5_Feature_Dataset.csv"
)

df = pd.read_csv(feature_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

print("\nData types:")
print(df.dtypes)

Dataset loaded successfully
Shape: (180, 15)

Columns:
['Company Name', 'Industry / Sub-industry', 'Founded Year (Standardized)', 'Target Customers / B2B or B2C', 'Employee Count (Numeric)', 'Company Size Score', 'B2B Fit Score', 'Technology Signal Score', 'Business Need Score', 'Recent Business Signal Score', 'Decision-Maker Score', 'Data Quality Status', 'Employee Validation', 'Company Age (Years)', 'Company Age Validation']

First five records:


,Company Name,Industry / Sub-industry,Founded Year (Standardized),Target Customers / B2B or B2C,Employee Count (Numeric),Company Size Score,B2B Fit Score,Technology Signal Score,Business Need Score,Recent Business Signal Score,Decision-Maker Score,Data Quality Status,Employee Validation,Company Age (Years),Company Age Validation
0,Postman,API Development / B2B SaaS,2014,B2B / developers and enterprises,750,20,20,20,15,15,10,Valid,Valid,12,Valid
1,BrowserStack,Software Testing / DevOps SaaS,2011,"B2B / developers, QA teams and enterprises",3000,20,20,20,5,15,10,Valid,Valid,15,Valid
2,Freshworks,SaaS / Business Software / AI,2010,B2B / businesses and enterprises,5000,20,20,20,15,5,10,Valid,Valid,16,Valid
3,Chargebee,SaaS / Subscription Billing,2011,B2B / SaaS and subscription businesses,3000,20,20,7,5,15,10,Valid,Valid,15,Valid
4,CleverTap,Customer Engagement SaaS / MarTech,2013,B2B / consumer brands and enterprises,750,20,20,15,15,15,10,Valid,Valid,13,Valid



Data types:
Company Name                     object
Industry / Sub-industry          object
Founded Year (Standardized)       int64
Target Customers / B2B or B2C    object
Employee Count (Numeric)          int64
Company Size Score                int64
B2B Fit Score                     int64
Technology Signal Score           int64
Business Need Score               int64
Recent Business Signal Score      int64
Decision-Maker Score              int64
Data Quality Status              object
Employee Validation              object
Company Age (Years)               int64
Company Age Validation           object
dtype: object


In [2]:
# Day 6 - Step 3: Audit available source columns

source_path = project_dir / "data" / "NexziByte_Final_Scored.csv"
clean_path = project_dir / "data" / "clean" / "Day3_Prospects_Clean.csv"

print("Original source exists:", source_path.exists())
print("Cleaned dataset exists:", clean_path.exists())

if source_path.exists():
    source_df = pd.read_csv(source_path)
    print("\nOriginal source shape:", source_df.shape)
    print("\nOriginal source columns:")
    print(source_df.columns.tolist())

if clean_path.exists():
    clean_df = pd.read_csv(clean_path)
    print("\nCleaned dataset shape:", clean_df.shape)
    print("\nCleaned dataset columns:")
    print(clean_df.columns.tolist())

    outcome_terms = [
        "convert", "qualified", "lead status", "outcome",
        "won", "lost", "accepted", "revenue", "closed"
    ]

    matching_columns = [
        col for col in clean_df.columns
        if any(term in col.lower() for term in outcome_terms)
    ]

    print("\nPotential outcome-related columns:")
    print(matching_columns if matching_columns else "None found")

Original source exists: False
Cleaned dataset exists: True

Cleaned dataset shape: (180, 28)

Cleaned dataset columns:
['Company Name', 'Website / Domain', 'Industry / Sub-industry', 'HQ Location (City, State)', 'Founded Year (Standardized)', 'Products/Services', 'Target Customers / B2B or B2C', 'Technology Signals', 'Recent Business Signal', 'Decision-Maker (Name + Title)', 'Source(s) Used', 'Date Verified', 'Employee Count (Numeric)', 'Website Validation', 'Employee Validation', 'Duplicate Flag', 'Data Quality Status', 'Company Size Score', 'B2B Fit Score', 'Technology Signal Score', 'Business Need Score', 'Recent Business Signal Score', 'Decision-Maker Score', 'Prospect Score', 'Prospect Classification', 'ETL Website Format Valid', 'ETL Company Name Unique', 'ETL Score Consistent']

Potential outcome-related columns:
None found


In [3]:
# Day 6 - Step 4: Create synthetic demonstration labels

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Copy the feature dataset; do not modify the source files
model_df = df.copy()

# Fixed random seed makes this demonstration reproducible
rng = np.random.default_rng(42)

# Synthetic labels are randomly generated for workflow demonstration only
model_df["Synthetic_Lead_Conversion"] = rng.binomial(
    n=1,
    p=0.35,
    size=len(model_df)
)

print("Modeling dataset shape:", model_df.shape)
print("\nSynthetic outcome distribution:")
print(
    model_df["Synthetic_Lead_Conversion"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Synthetic Not Converted", 1: "Synthetic Converted"})
)

print("\nIMPORTANT:")
print("These labels are synthetic and have no verified business meaning.")
print("Do not interpret resulting model metrics as actual predictive performance.")

Modeling dataset shape: (180, 16)

Synthetic outcome distribution:
Synthetic_Lead_Conversion
Synthetic Not Converted    117
Synthetic Converted         63
Name: count, dtype: int64

IMPORTANT:
These labels are synthetic and have no verified business meaning.
Do not interpret resulting model metrics as actual predictive performance.


In [4]:
# Day 6 - Step 5: Prepare train/test split

# Exclude identifiers, existing component scores, and data-quality flags
# to avoid learning the existing scoring system instead of conversion.
categorical_features = [
    "Industry / Sub-industry",
    "Target Customers / B2B or B2C"
]

numeric_features = [
    "Founded Year (Standardized)",
    "Employee Count (Numeric)",
    "Company Age (Years)"
]

feature_cols = categorical_features + numeric_features
target_col = "Synthetic_Lead_Conversion"

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

print("\nFeatures used:", feature_cols)
print("Target:", target_col)
print("Target type: SYNTHETIC — demonstration only")

Training features: (135, 5)
Testing features: (45, 5)

Training target distribution:
Synthetic_Lead_Conversion
0    88
1    47
Name: count, dtype: int64

Testing target distribution:
Synthetic_Lead_Conversion
0    29
1    16
Name: count, dtype: int64

Features used: ['Industry / Sub-industry', 'Target Customers / B2B or B2C', 'Founded Year (Standardized)', 'Employee Count (Numeric)', 'Company Age (Years)']
Target: Synthetic_Lead_Conversion
Target type: SYNTHETIC — demonstration only


In [5]:
# Day 6 - Step 6: Logistic Regression baseline

# Preprocessing for numeric and categorical features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fit only on training data
baseline_model.fit(X_train, y_train)

# Generate test predictions
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Logistic Regression baseline trained.")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Predictions generated:", len(y_pred))
print("\nReminder: target labels are synthetic.")

Logistic Regression baseline trained.
Training records: 135
Testing records: 45
Predictions generated: 45

Reminder: target labels are synthetic.


In [6]:
# Day 6 - Step 7: Evaluate synthetic baseline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC (Average Precision)": average_precision_score(y_test, y_prob)
}

metrics_df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "Value"]
)

print("SYNTHETIC DEMONSTRATION — NOT REAL BUSINESS PERFORMANCE\n")
print(metrics_df.to_string(index=False))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Synthetic Not Converted", "Synthetic Converted"],
    zero_division=0
))

SYNTHETIC DEMONSTRATION — NOT REAL BUSINESS PERFORMANCE

                    Metric    Value
                  Accuracy 0.622222
                 Precision 0.333333
                    Recall 0.062500
                  F1 Score 0.105263
                   ROC-AUC 0.517241
PR-AUC (Average Precision) 0.421671

Confusion Matrix:
[[27  2]
 [15  1]]

Classification Report:
                         precision    recall  f1-score   support

Synthetic Not Converted       0.64      0.93      0.76        29
    Synthetic Converted       0.33      0.06      0.11        16

               accuracy                           0.62        45
              macro avg       0.49      0.50      0.43        45
           weighted avg       0.53      0.62      0.53        45



In [7]:
# Day 6 - Step 8: Save synthetic model outputs

from datetime import datetime

processed_dir = project_dir / "data" / "processed"
reports_dir = project_dir / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Save metrics
metrics_df["Data Status"] = "SYNTHETIC DEMONSTRATION ONLY"
metrics_path = processed_dir / "Day6_Synthetic_Baseline_Metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

# Save test-set predictions with clear synthetic labels
scored_output = X_test.copy()
scored_output["Synthetic_Actual_Label"] = y_test
scored_output["Synthetic_Predicted_Label"] = y_pred
scored_output["Synthetic_Predicted_Probability"] = y_prob
scored_output["Data Status"] = "SYNTHETIC DEMONSTRATION ONLY"

scored_output = scored_output.reset_index(names="Source_Row_Index")
scored_path = processed_dir / "Day6_Synthetic_Scored_Output.csv"
scored_output.to_csv(scored_path, index=False)

# Save a readable report
report_path = reports_dir / "Day6_Lead_Model_Evaluation.md"

report = f"""# Day 6 — Lead Model Evaluation

## Data status
SYNTHETIC DEMONSTRATION ONLY. Outcome labels were randomly generated.
These metrics do not represent actual NexziByte lead-conversion performance.

## Dataset
- Total records: {len(model_df)}
- Training records: {len(X_train)}
- Test records: {len(X_test)}
- Input features: {", ".join(feature_cols)}
- Target: Synthetic_Lead_Conversion

## Logistic Regression baseline

| Metric | Value |
|---|---:|
| Accuracy | {metrics["Accuracy"]:.4f} |
| Precision | {metrics["Precision"]:.4f} |
| Recall | {metrics["Recall"]:.4f} |
| F1-score | {metrics["F1 Score"]:.4f} |
| ROC-AUC | {metrics["ROC-AUC"]:.4f} |
| PR-AUC (Average Precision) | {metrics["PR-AUC (Average Precision)"]:.4f} |

## Confusion matrix
Rows are actual labels [0, 1]; columns are predicted labels [0, 1].

{confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()}

## Limitations
- No verified historical conversion outcome was available in the audited prospect dataset.
- Synthetic labels were randomly generated and do not represent real business outcomes.
- These metrics must not be used for sales decisions or presented as real model performance.
- A production lead propensity model requires a business-approved target definition,
  historical outcomes, an observation window, and a leakage-safe evaluation design.

## Outputs
- data/processed/Day6_Synthetic_Baseline_Metrics.csv
- data/processed/Day6_Synthetic_Scored_Output.csv
"""

report_path.write_text(report, encoding="utf-8")

print("Metrics saved:", metrics_path)
print("Scored output saved:", scored_path)
print("Evaluation report saved:", report_path)

print("\nScored output shape:", scored_output.shape)
print("Metrics rows:", len(metrics_df))

Metrics saved: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day6_Synthetic_Baseline_Metrics.csv
Scored output saved: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day6_Synthetic_Scored_Output.csv
Evaluation report saved: D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day6_Lead_Model_Evaluation.md

Scored output shape: (45, 10)
Metrics rows: 6


In [8]:
# Day 6 - Step 9: Create synthetic demonstration priority queue

priority_queue = scored_output.copy()

priority_queue["Demonstration Priority"] = (
    priority_queue["Synthetic_Predicted_Probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

priority_queue["Demonstration Priority Band"] = np.select(
    [
        priority_queue["Demonstration Priority"] <= 10,
        priority_queue["Demonstration Priority"] <= 25
    ],
    [
        "High - Demo Only",
        "Medium - Demo Only"
    ],
    default="Low - Demo Only"
)

priority_queue = priority_queue.sort_values(
    "Demonstration Priority"
)

priority_path = (
    project_dir / "data" / "processed"
    / "Day6_Synthetic_Priority_Queue.csv"
)

priority_queue.to_csv(priority_path, index=False)

print("Priority queue saved:", priority_path)
print("Queue shape:", priority_queue.shape)

display(priority_queue.head(10))

Priority queue saved: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day6_Synthetic_Priority_Queue.csv
Queue shape: (45, 12)


,Source_Row_Index,Industry / Sub-industry,Target Customers / B2B or B2C,Founded Year (Standardized),Employee Count (Numeric),Company Age (Years),Synthetic_Actual_Label,Synthetic_Predicted_Label,Synthetic_Predicted_Probability,Data Status,Demonstration Priority,Demonstration Priority Band
17,6,FinTech / Payments,B2B / businesses and enterprises,2014,4000,12,1,1,0.605860,SYNTHETIC DEMONSTRATION ONLY,1,High - Demo Only
19,8,FinTech / Payments,B2B & B2C / consumers and merchants,2015,5000,11,0,1,0.579250,SYNTHETIC DEMONSTRATION ONLY,2,High - Demo Only
15,98,AI / ML / GenAI,"B2B — large enterprises across BFSI, healthcar...",2000,5000,26,0,1,0.535929,SYNTHETIC DEMONSTRATION ONLY,3,High - Demo Only
8,169,FinTech / Payments,B2B — SMEs and businesses purchasing raw mater...,2016,3000,10,1,0,0.455435,SYNTHETIC DEMONSTRATION ONLY,4,High - Demo Only
4,154,FinTech / Payments,B2C — consumers seeking digital banking and mo...,2019,500,7,0,0,0.430414,SYNTHETIC DEMONSTRATION ONLY,5,High - Demo Only
21,146,FinTech / Payments,"B2B — merchants, banks, fintechs and large dig...",2012,2000,14,0,0,0.416415,SYNTHETIC DEMONSTRATION ONLY,6,High - Demo Only
22,105,AI / ML / GenAI,"B2B — advertisers, brands, app developers and ...",2005,2500,21,1,0,0.369214,SYNTHETIC DEMONSTRATION ONLY,7,High - Demo Only
32,160,FinTech / Payments,"B2B — banks, NBFCs, fintechs and lending insti...",2020,1500,6,0,0,0.353130,SYNTHETIC DEMONSTRATION ONLY,8,High - Demo Only
44,133,AI / ML / GenAI,B2C + B2B2C — riders/passengers and driver-par...,2015,3000,11,0,0,0.352759,SYNTHETIC DEMONSTRATION ONLY,9,High - Demo Only
14,162,FinTech / Payments,B2B — SMEs and small-business owners,2014,1000,12,0,0,0.352212,SYNTHETIC DEMONSTRATION ONLY,10,High - Demo Only


In [9]:
# Day 6 - Final output verification

output_files = [
    project_dir / "data" / "processed"
    / "Day6_Synthetic_Baseline_Metrics.csv",

    project_dir / "data" / "processed"
    / "Day6_Synthetic_Scored_Output.csv",

    project_dir / "data" / "processed"
    / "Day6_Synthetic_Priority_Queue.csv",

    project_dir / "reports"
    / "Day6_Lead_Model_Evaluation.md"
]

for path in output_files:
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\nPriority queue label:")
print(priority_queue["Data Status"].unique())

print("\nScored records:", len(scored_output))
print("Priority queue records:", len(priority_queue))
print("Synthetic target only:", target_col == "Synthetic_Lead_Conversion")

Day6_Synthetic_Baseline_Metrics.csv: EXISTS
Day6_Synthetic_Scored_Output.csv: EXISTS
Day6_Synthetic_Priority_Queue.csv: EXISTS
Day6_Lead_Model_Evaluation.md: EXISTS

Priority queue label:
['SYNTHETIC DEMONSTRATION ONLY']

Scored records: 45
Priority queue records: 45
Synthetic target only: True
